# 03. Do cells group by cell type, or by donor and batch?

From the normalized matrix to clusters. The question is the same one 05-airway asked about samples: what is the biggest source of variation in this data?

**First move, in the terminal:** `uv add igraph leidenalg "harmonypy<0.1"`. igraph and leidenalg run the clustering, harmonypy runs batch correction in C6. The `<0.1` pin matters: harmonypy 2.x has to be compiled with C++ and fails on your laptop without a compiler; 0.0.x is pure Python.

**Input:** `C:/bio_data/ra_pbmc_qc.h5ad` · **Output:** `C:/bio_data/ra_pbmc_clustered.h5ad`

> **How this notebook works.** Each step has a **spec** (what the code must do), a **read-first** link, an empty code cell for you, and a **TEST** cell written by Claude. Your cell is done when its test prints PASS and you can say why your approach works. The ✍️ cells are yours to write; predictions go in *before* you run the cell they predict.
>
> No function names are given on purpose. Find them in the read-first link or the scanpy docs. Stuck for more than 10 minutes on one cell: ask Claude for a *question*, not code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

In [ ]:
# adata = the QC'd file from 02

...

## C1. Highly variable genes

**Spec:** mark the 2,000 most variable genes. Print the top 20 by dispersion.

**Read first:** https://www.sc-best-practices.org/preprocessing_visualization/feature_selection.html

✍️ **Prediction: which kinds of genes will top the list in blood? (Think about which genes are on in one cell type and off in all others.)**

_..._

In [ ]:
# mark 2000 HVGs, show the top 20

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert adata.var["highly_variable"].sum() == 2000
print("PASS: 2000 HVGs")

✍️ **Were you right? Name three genes from the top 20 and the cell type each belongs to.**

_..._

## C2. PCA

**Spec:** PCA with 50 components on the HVGs only. Plot the variance ratio per component (the elbow plot). Pick `n_pcs` and justify it in one sentence. You did PCA by hand in 05-airway: same idea, now 108k rows instead of 8.

**Read first:** https://www.sc-best-practices.org/preprocessing_visualization/dimensionality_reduction.html

In [ ]:
# PCA on HVGs, elbow plot
# n_pcs = your choice

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert adata.obsm["X_pca"].shape == (adata.n_obs, 50)
assert 10 <= n_pcs <= 50
print("PASS: PCA done, using", n_pcs, "PCs")

✍️ **Why that number of PCs, and what is the cost of picking too few or too many?**

_..._

## C3. Neighbours, UMAP, Leiden

**Spec:** build the neighbour graph on your `n_pcs`, then UMAP, then Leiden clustering (igraph flavour) into `obs['leiden']`. Start at resolution 1.0. This is the slowest step in the project (a few minutes on your laptop).

**Read first:** https://www.sc-best-practices.org/cellular_structure/clustering.html

✍️ **Prediction: how many clusters at resolution 1.0? The authors found 18 subsets.**

_..._

In [ ]:
# neighbours on n_pcs, UMAP, Leiden into obs['leiden']

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
k = adata.obs["leiden"].nunique()
assert 8 <= k <= 45, f"{k} clusters: check resolution"
assert "X_umap" in adata.obsm
print("PASS:", k, "clusters")

## C4. Write a purity score

**Spec:** write `purity(clusters, labels)`: for each cluster, the share of its cells that carry the cluster's most common label, averaged over all cells (weight by cluster size). 1.0 means every cluster holds one label only. Write it by hand (a crosstab gets you most of the way).

In [ ]:
# def purity(clusters, labels): ...

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
a = pd.Series(["0", "0", "0", "1", "1", "1"])
b = pd.Series(["T", "T", "B", "B", "B", "B"])
assert np.isclose(purity(a, b), 5 / 6), purity(a, b)
assert np.isclose(purity(b, b), 1.0)
print("PASS: purity works on the known case")

## C5. Cell type or batch?

**Spec:** four UMAPs side by side, coloured by `leiden`, the authors' `fine_annot`, `batch`, and `disease`. Then compute purity of your clusters against `rough_annot`, `fine_annot`, `batch` and `donor_id`.

✍️ **Prediction first: which of the four will your clusters be most pure for, and which least?**

_..._

In [ ]:
# 4 UMAPs
# purity against rough_annot, fine_annot, batch, donor_id

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
p_rough_before = purity(adata.obs["leiden"], adata.obs["rough_annot"])
p_batch_before = purity(adata.obs["leiden"], adata.obs["batch"].astype(str))
baseline = adata.obs["batch"].value_counts(normalize=True).max()
print(f"RECORDED (no pass bar yet): rough {p_rough_before:.3f} | batch {p_batch_before:.3f} | batch baseline {baseline:.3f}")

✍️ **Do you need batch correction? Answer with the numbers, not an opinion. The baseline is what batch purity would be if batch did not drive the clusters at all (every cluster mixed like the whole dataset).**

_..._

## C6. Batch correction with Harmony

**Spec:** run Harmony on your PCA with `batch` as the key. It writes a corrected embedding into `obsm`. Redo neighbours, UMAP and Leiden **on that embedding**. Keep the old clusters as `obs['leiden_uncorrected']` first, so you can compare.

**Read first:** https://www.sc-best-practices.org/cellular_structure/integration.html (intro and the Harmony part). The key idea: Harmony moves cells in PCA space so that batches mix *within* each cell type, not across them.

✍️ **Prediction: what happens to batch purity and rough purity after correction?**

_..._

In [ ]:
# obs['leiden_uncorrected'] = copy of obs['leiden']
# Harmony on batch -> corrected embedding
# neighbours (use_rep = that embedding), UMAP, Leiden again into obs['leiden']

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
p_rough = purity(adata.obs["leiden"], adata.obs["rough_annot"])
p_batch = purity(adata.obs["leiden"], adata.obs["batch"].astype(str))
assert p_batch < 0.55, f"batch purity {p_batch:.3f}: batch still drives the clusters"
assert p_rough > 0.80, f"rough purity {p_rough:.3f}"
print(f"PASS: rough {p_rough_before:.3f} -> {p_rough:.3f} | batch {p_batch_before:.3f} -> {p_batch:.3f}")

✍️ **Redo the four UMAPs. What changed? The authors' own PCA reaches about 0.97 rough purity, so Harmony alone does not close the gap. What else did they do differently? (Hint: 01 E2 part 3. Try leaving those genes out of the HVGs and rerun.)**

_..._

✍️ **Stretch: rerun Leiden at 0.5 and 2.0. Which resolution recovers the authors' fine labels best, and why does purity alone always prefer more clusters?**

_..._

## C7. Save

**Spec:** write to `C:/bio_data/ra_pbmc_clustered.h5ad`.

In [ ]:
# write the file

...

## Teach-back

3 to 5 sentences in your own words: what this notebook did, what it found, and one thing that surprised you. No sentence means not learned.

_..._

## AI note

What did Claude write (the specs, the tests), what did you write, and where did you get stuck and ask for a hint?

_..._